# 04. 시뮬레이션 시장과 민감도

오늘 시장(`TodaysMarket`)은 바뀌지 않는다. 흔들기는 같은 규격의 **시뮬레이션 시장**이 맡는다(원칙 1, 7).

| 단계 | QuantTool 코드 | ORE 원본 (`references/Engine/OREAnalytics/orea/`) |
|---|---|---|
| 격자 | `configs/scenario/simmarket.yaml` → `ScenarioSimMarketParameters` | `scenario/scenariosimmarketparameters.hpp`, `simulation.xml` |
| 리스크 요인·시나리오 | `qle/scenario.py` `RiskFactorKey`, `SimpleScenario`, `DeltaScenario` | `QuantExt/qle/termstructures/scenario.hpp`, `scenario/deltascenario.hpp` |
| 시뮬레이션 시장 | `analytics/scenario/simmarket.py` | `simulation/simmarket.hpp`, `scenario/scenariosimmarket.cpp` |
| 생성기 | `analytics/scenario/sensitivityscenariogenerator.py`, `shiftscenariogenerator.py` | `scenario/sensitivityscenariogenerator.cpp`, `shiftscenariogenerator.cpp` |
| 큐브·반복 평가 | `analytics/cube/`, `analytics/engine/valuationengine.py` | `cube/npvcube.hpp`, `engine/valuationengine.hpp`, `valuationcalculator.hpp` |
| 민감도 | `analytics/engine/sensitivityanalysis.py`, `analytics/cube/sensitivitycube.py` | `engine/sensitivityanalysis.cpp`, `cube/sensitivitycube.cpp` |

> 데이터: `tests/fixtures/market_ust_sample.txt`는 ORE `market.txt` 형식(`날짜 키 값`)의 **합성 샘플**이다. 흐름을 익히는 용도이고, 숫자로 시장을 해석하지 않는다.

In [ ]:
from pathlib import Path

# 노트북이 어디서 열리든 레포 루트(pyproject.toml이 있는 곳)를 찾는다
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
SAMPLE = ROOT / "tests" / "fixtures" / "market_ust_sample.txt"  # 합성 샘플 (실제 시세 아님)

from datetime import date

import pandas as pd
import QuantLib as ql

from quanttool.analytics.cube import NPVCube
from quanttool.analytics.engine import NPVCalculator, SensitivityAnalysis, ValuationEngine
from quanttool.analytics.scenario import (
    KeyType,
    ScenarioSimMarket,
    SensitivityScenarioGenerator,
    zero_rate_shift_scenario,
)
from quanttool.data.configuration import load_config
from quanttool.data.marketdata import CSVLoader, TodaysMarket
from quanttool.data.portfolio import EngineFactory, Portfolio

cfg = load_config(ROOT / "configs")
market = TodaysMarket.from_config(date(2026, 10, 23), cfg, CSVLoader(SAMPLE))
book = Portfolio.from_file(ROOT / "positions" / "ust_book.yaml")

## 1. 격자: 흔들 수 있는 좌표
격자점 하나가 `RiskFactorKey(KeyType.YieldCurve, 'UST', 번호)` 하나다. PCA 로딩이나 과거 변화도 이 좌표로 정의한다(원칙 7-1).

In [ ]:
params = cfg.simmarket
tenors = params.yield_curve_tenors("UST")
keys = params.keys(KeyType.YieldCurve, "UST")
print(params.curves(), params.interpolation, params.extrapolation)
pd.DataFrame({"key": [str(k) for k in keys], "tenor": tenors}).head()

## 2. 시뮬레이션 시장: spreaded vs 절대 모드
- **spreaded**(민감도 설정의 기본): 격자 Quote = 초기 커브 대비 **할인계수 비율**(기준 1.0). 기준 상태에서 초기 커브와 **정확히** 같다(ORE `SpreadedDiscountCurve`).
- **절대**: 격자 Quote = 할인계수 자체. 격자점 사이를 제로금리 선형 보간하므로 초기 커브와 조금 다르다(ORE `InterpolatedDiscountCurve`).

In [ ]:
sim = ScenarioSimMarket(market, params, use_spreaded_term_structures=True)
sim_abs = ScenarioSimMarket(market, params, use_spreaded_term_structures=False)
k5 = keys[tenors.index("5Y")]
print(
    "기준 시나리오 값 (5Y): spreaded",
    sim.base_scenario().get(k5),
    " 절대",
    sim_abs.base_scenario().get(k5),
)
rows = []
with sim.active():
    for y in [0.5, 1.25, 2.5, 11.0, 27.0]:
        z0 = market.yield_curve("UST").zeroRate(y, ql.Continuous).rate()
        rows.append(
            {
                "years": y,
                "spreaded_diff_bp": (sim.yield_curve("UST").zeroRate(y, ql.Continuous).rate() - z0)
                * 1e4,
                "absolute_diff_bp": (
                    sim_abs.yield_curve("UST").zeroRate(y, ql.Continuous).rate() - z0
                )
                * 1e4,
            }
        )
pd.DataFrame(rows).set_index("years").round(6)

In [ ]:
display(sim.base_reconciliation(book, cfg))
sim_abs.base_reconciliation(book, cfg)

## 3. 시나리오를 손으로 만들어 적용하기
시나리오 = 격자점 → **값**(이 시장에서는 할인계수 비율). `zero_rate_shift_scenario`가 "이 격자점 제로금리를 이만큼"을 값으로 바꿔 `DeltaScenario`(기준 + 바뀐 점)를 만든다. 포트폴리오는 시뮬레이션 시장에 **한 번** 조립해 둔다.

In [ ]:
with sim.active():
    book.build(EngineFactory.from_config(cfg, sim))
    base = book.npvs()
    s = zero_rate_shift_scenario(sim, {k: 0.0010 for k in keys}, "parallel +10bp")
    print(s.label(), " 5Y 값", s.get(k5))
    sim.apply_scenario(s)
    parallel = book.npvs() - base
    sim.reset()
parallel.round(2)

### 예시: 스티프닝 (PCA 요인 시나리오의 모양)
PCA 시나리오는 "요인 크기 × 격자점별 로딩"이다. 아래 로딩은 **설명용으로 만든 값**(단기 −5bp → 장기 +5bp 선형)이고 추정값이 아니다.

In [ ]:
times = [sim.sim_data_times[k] for k in keys]
loading = {k: -1 + 2 * t / times[-1] for k, t in zip(keys, times, strict=True)}  # −1 … +1
with sim.active():
    sim.apply_scenario(
        zero_rate_shift_scenario(sim, {k: 0.0005 * w for k, w in loading.items()}, "steepener")
    )
    steep = book.npvs() - base
    sim.reset()
steep.round(2)

## 4. 생성기 + ValuationEngine + NPVCube
민감도 생성기는 기준 + shift tenor마다 Up(+1bp), gamma용 Down(−1bp) 시나리오를 만든다. 시나리오 설명이 무엇을 흔들었는지 기록한다. `ValuationEngine.build_cube`는 표본(시나리오)마다·날짜마다 `sim.update(d)` → 계산기(`NPVCalculator`) → 큐브에 기록한다.

In [ ]:
gen = SensitivityScenarioGenerator(cfg.sensitivity, sim.base_scenario(), params, sim)
print("시나리오 수", gen.samples(), " 예:", [str(d) for d in gen.scenario_descriptions()[:3]])
sim.scenario_generator = gen
cube = NPVCube(book.ids(), sim.asof, [sim.asof], gen.samples())
ValuationEngine(sim.asof, [sim.asof], sim).build_cube(book, cube, [NPVCalculator("USD")])
frame = cube.to_frame(labels=[str(d) for d in gen.scenario_descriptions()])
(frame - cube.t0()).head()

## 5. SensitivityAnalysis: 위 과정을 ORE 순서대로 한 번에
시뮬레이션 시장 구축 → 템플릿별 생성기 → 포트폴리오를 시뮬레이션 시장에 연결 → ValuationEngine → `SensitivityCube` (ORE `SensitivityAnalysis::generateSensitivities`). 결과는 ORE와 같이 **설정한 이동 1회(1bp)당 NPV 변화**다.

In [ ]:
sa = SensitivityAnalysis.from_config(book, market, cfg)
sa.generate_sensitivities()
display(sa.report().head())
krd = sa.delta_table(KeyType.YieldCurve, "UST")
with sim.active():
    book.build(EngineFactory.from_config(cfg, sim))
    sim.apply_scenario(zero_rate_shift_scenario(sim, {k: 1e-4 for k in keys}, "parallel +1bp"))
    par1 = (book.npvs() - base).sum()
    sim.reset()
print(
    "Σ key rate delta",
    round(krd["total"].sum(), 2),
    " / 평행 +1bp 시나리오",
    round(par1, 2),
    "(차이 = 2차 항)",
)
krd.round(1)

In [ ]:
sa.delta_table(KeyType.YieldCurve, "UST", measure="Gamma").round(4).tail()

## 6. 거래별 민감도 규칙: SensitivityTemplate
엔진 파라미터 `SensitivityTemplate`을 단 거래만 다른 이동 크기·방식으로 계산할 수 있다(ORE). 아래는 10bp 중심 차분으로 계산하고 1bp 기준으로 다시 환산한 것이다. 상품별로 그릭 계산 규칙을 바꾸는 공식 경로다.

In [ ]:
ed = cfg.engine_data.model_copy(deep=True)
ed.products["Bond"] = ed.products["Bond"].model_copy(
    update={"engine_parameters": {"SensitivityTemplate": "IR_FD"}}
)
ust = cfg.sensitivity.yield_curves["UST"].model_copy(
    update={"keyed_shift_size": {"IR_FD": 0.001}, "keyed_shift_scheme": {"IR_FD": "Central"}}
)
data = cfg.sensitivity.model_copy(update={"yield_curves": {"UST": ust}})
book.reset()
sa_fd = SensitivityAnalysis(book, market, "default", ed, params, data)
sa_fd.generate_sensitivities()
pd.DataFrame(
    {
        "1bp Forward": krd["UST_10Y"],
        "10bp Central → 1bp": sa_fd.delta_table(KeyType.YieldCurve, "UST")["UST_10Y"],
    }
).round(2).loc[["5Y", "10Y"]]

## 정리
- 시나리오 = 격자 좌표 위의 값. 규칙(민감도·스트레스·PCA·과거 변화)만 다르고 적용 방식은 같다.
- 지금 delta는 **격자 제로금리 기준**이다. 헷지 수량(국채 par 기준)으로 바꾸는 것이 `ParSensitivityAnalysis`(다음 단계).
- 다음: `05_time_and_history.ipynb` — 시간을 움직이고, 날짜별 시장을 표로 쌓는다.